In [1]:
!pip install transformers seqeval evaluate datasets

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/10.5 MB ? eta -:--:--
   ----------------------- ---------------- 6.0/10.5 MB 30.7 MB/s eta 0:00:01
   ---------------------------------------- 10.5/10.5 MB 27.2 MB/s  0:00:00
   ---------------------------------------- 0.0/647.8 kB ? eta -:--:--
   ---------------------------------------- 647.8/647.8 kB 6.2 MB/s  0:00:00
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ---------------------------------------- 3.7/3.7 MB 19.9 MB/s  0:00:00
   ---------------------------------------- 0.0/2


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Data Loading

In [2]:
from datasets import load_dataset
dataset_raw = load_dataset("lfcc/portuguese_ner")
dataset_raw


c:\Users\barba\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\barba\.cache\huggingface\hub\datasets--lfcc--portuguese_ner. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating test split: 100%|██████████| 930/930 [00:00<00:00, 37041.60 examples/s]


DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 3716
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 930
    })
})

In [3]:
dataset_raw["train"].features

{'tokens': List(Value('string')),
 'ner_tags': List(ClassLabel(names=['O', 'B-Data', 'I-Data', 'B-Local', 'I-Local', 'B-Organizacao', 'I-Organizacao', 'B-Pessoa', 'I-Pessoa', 'B-Profissao', 'I-Profissao']))}

Data Pre-Processing

In [4]:
from transformers import AutoTokenizer

tokenizer=AutoTokenizer.from_pretrained("neuralmind/bert-base-portuguese-cased")

c:\Users\barba\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\barba\.cache\huggingface\hub\models--neuralmind--bert-base-portuguese-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [6]:
inputs=tokenizer("As aulas de PLNEB são muito interessantes!")
inputs

{'input_ids': [101, 510, 6880, 125, 212, 22327, 22320, 19591, 453, 785, 20764, 106, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [ ]:
tokens=tokenizer.convert_ids_to_tokens(inputs["input_ids"])
print(tokens)

###[SEP] é um token especial que indica o fim de uma sequência de texto. Ele é usado para separar diferentes partes do texto, como frases ou parágrafos, e também pode ser usado para indicar o final de uma entrada em tarefas de processamento de linguagem natural, como classificação ou geração de texto. O token [SEP] é importante para ajudar os modelos de linguagem a entenderem a estrutura do texto e a processarem as informações de maneira adequada.
###[CLS] é um token especial que é adicionado no início de uma sequência de texto em tarefas de processamento de linguagem natural, como classificação ou geração de texto. Ele é usado para indicar o início da entrada e é importante para ajudar os modelos de linguagem a entenderem a estrutura do texto e a processarem as informações de maneira adequada. O token [CLS] é frequentemente usado em conjunto com o token [SEP] para separar diferentes partes do texto e indicar o início e o fim da entrada.

['[CLS]', 'As', 'aulas', 'de', 'P', '##L', '##N', '##EB', 'são', 'muito', 'interessantes', '!', '[SEP]']


In [8]:
dataset_raw["train"]["tokens"]

Column([['Filiação', ':', 'Antonio', 'Joaquim', 'Aguiar', 'e', 'Engracia', 'Maria', '.', 'Natural', 'e/ou', 'residente', 'em', 'CUNHA', ',', 'Santa', 'Maria', ',', 'actual', 'concelho', 'de', 'PAREDES', 'COURA', 'e', 'distrito', '(', 'ou', 'país', ')', 'Viana', 'do', 'Castelo', '.'], ['Filiação', ':', 'Domingos', 'Pires', 'e', 'Comba', 'Fernandes', '.', 'Natural', 'e/ou', 'residente', 'em', 'VALONGO', 'MILHAIS', ',', 'Sao', 'Goncalo', ',', 'actual', 'concelho', 'de', 'MURCA', 'e', 'distrito', '(', 'ou', 'país', ')', 'VILA', 'REAL', '.'], ['Termo', 'de', 'justificação', 'do', 'baptismo', 'de', 'Pedro', 'Gonçalves', 'Coques', ',', 'nascido', 'em', '29.06.1876', 'e', 'baptizado', '"', '(', '…', ')', 'por', 'dias', 'do', 'mês', 'de', 'Julho', 'do', 'dito', 'ano', ',', '(', '…', ')', '"', ',', 'na', 'igreja', 'do', 'Jardim', 'do', 'Mar', ',', 'Calheta', '.'], ['Doc.danificado', '.'], ['1898-11-01', '/', '1898-11-01'], ...])

In [11]:
tokens = ["as", "aulas", "de", "plneb", "são","interessantes", "!"]
inputs = tokenizer(tokens, is_split_into_words=True) ##aviso de que os tokens já estão separados, para evitar que o tokenizer tente dividir as palavras novamente.

new_tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"])
print(new_tokens)

['[CLS]', 'as', 'aulas', 'de', 'pl', '##ne', '##b', 'são', 'interessantes', '!', '[SEP]']


In [12]:
len(tokens), len(new_tokens)

(7, 11)

In [13]:
inputs.word_ids() ##mapeamento direto entre os tokens e as palavras originais. Cada token é associado a um índice que indica a qual palavra original ele pertence. Se um token for parte de uma palavra original, ele terá o mesmo índice que essa palavra. Se um token for um token especial, como [CLS] ou [SEP], ele terá um índice diferente. O método word_ids() é útil para alinhar as etiquetas de uma tarefa de processamento de linguagem natural com os tokens gerados pelo tokenizer, garantindo que as etiquetas sejam atribuídas corretamente aos tokens correspondentes.

[None, 0, 1, 2, 3, 3, 3, 4, 5, 6, None]

In [16]:
def align_labels_with_tokens(word_ids, labels):
    new_labels = []
    previous_word = None

    for word_id in word_ids:
        if word_id is None:
            new_labels.append(-100)  # código que o BERT usa para ignorar tokens especiais
        elif word_id != previous_word:
            new_labels.append(labels[word_id])  # etiqueta da palavra original
        else:
            new_labels.append(-100)  # código para ignorar sub-tokens
        previous_word = word_id

    return new_labels


def tokenize_dataset(dataset):
    res=[]
    for row in dataset:
        inputs=tokenizer(row["tokens"], is_split_into_words=True) ##vai me dar os meus inputs, mas as labels estao mal entao temos de aplicar a função align_labels_with_tokens para corrigir as labels
        new_labels=align_labels_with_tokens(inputs.word_ids(), row["ner_tags"]) ##vai me dar as minhas labels corrigidas, alinhadas com os tokens gerados pelo tokenizer
        inputs["labels"]=new_labels ##adiciona as labels corrigidas aos meus inputs
        res.append(inputs) ##adiciona os meus inputs corrigidos a uma lista de resultados
    return res


train_dataset = tokenize_dataset(dataset_raw["train"])
test_dataset = tokenize_dataset(dataset_raw["test"])

len(train_dataset), len(test_dataset)



(3716, 930)

In [17]:
from datasets import Dataset
train_dataset = Dataset.from_list(train_dataset)
test_dataset = Dataset.from_list(test_dataset)

print(train_dataset)
print(test_dataset)

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 3716
})
Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 930
})


Model Training

In [20]:
from transformers import AutoModelForTokenClassification
model=AutoModelForTokenClassification.from_pretrained("neuralmind/bert-base-portuguese-cased")

ImportError: 
AutoModelForTokenClassification requires the PyTorch library but it was not found in your environment. Check out the instructions on the
installation page: https://pytorch.org/get-started/locally/ and follow the ones that match your environment.
Please note that you may need to restart your runtime after installation.
